In [1]:
import os
from pathlib import Path
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

model = ChatOpenAI(
    model=os.getenv("OLLAMA_MODEL"),
    base_url=f"{os.getenv('OLLAMA_BASE_URL')}/v1",
    api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
    temperature=0,
)


In [2]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [3]:
def create_outline(state: BlogState)-> BlogState:
    title= state['title']

    prompt= f"Generate a detailed outline for a blog on the topic: {title}"
    outline= model.invoke(prompt)

    state['outline']= outline

    return state


In [8]:
def create_blog(state: BlogState)-> BlogState:
    title= state['title']
    outline= state['outline']

    prompt= f"Write a detailed blog on title -{title} using the following outline \n {outline}"
    content= model.invoke(prompt)
    state['content']= content
    return state

In [10]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [11]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': AIMessage(content='Certainly! Here\'s a detailed outline for a blog titled "The Rise of AI in India":\n\n### Title: The Rise of AI in India\n\n#### Introduction\n- **Hook**: Start with an intriguing fact or statistic about AI adoption in India.\n- **Background**: Brief overview of Artificial Intelligence and its significance globally.\n- **Thesis Statement**: Highlight the growing importance and impact of AI in India.\n\n#### Chapter 1: Understanding AI in India\n- **Definition and Scope**:\n  - What is AI?\n  - Types of AI (Narrow, General, Superintelligence).\n- **Current State of AI in India**:\n  - Overview of AI adoption across sectors.\n  - Key players and startups in the Indian AI landscape.\n\n#### Chapter 2: Government Initiatives and Policies\n- **National Artificial Intelligence Strategy**:\n  - Details on the National Strategy for AI announced by the government.\n  - Objectives and goals set forth in the strategy.\n- **Government 